# Orbit Wars Local Benchmark

Notebook này dùng để chạy benchmark Orbit Wars ngay trên máy local.

Cách dùng nhanh:
1. Chạy `setup_local_env.ps1` một lần để tạo `.venv` trong thư mục dự án.
2. Chọn kernel `Python (Orbit Wars .venv)` trong VS Code hoặc Jupyter.
3. Chạy các cell từ trên xuống.
4. Replay sẽ được lưu dưới dạng JSON gọn kèm một HTML summary nhẹ trong thư mục `replays/`.

Ghi chú:
- Không render trực tiếp bằng `mode="ipython"` để tránh làm notebook lag.
- Mặc định dùng `debug=False` để trận chạy nhẹ hơn.
- Các file agent nằm cùng thư mục với notebook.


In [10]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
VENV_DIR = ROOT / ".venv"
REPLAY_DIR = ROOT / "replays"

print("Project root:", ROOT)
print("Python:", sys.executable)

if not VENV_DIR.exists():
    print("Chưa thấy .venv. Hãy chạy: powershell -ExecutionPolicy Bypass -File ./setup_local_env.ps1")
elif VENV_DIR.resolve() not in Path(sys.executable).resolve().parents:
    print("Bạn chưa chọn kernel .venv của project. Hãy chọn: Python (Orbit Wars .venv)")
else:
    print("Đang dùng đúng .venv của project.")


Project root: D:\Orbit Wars
Python: d:\Orbit Wars\.venv\Scripts\python.exe
Đang dùng đúng .venv của project.


In [11]:
import logging
import os
import shutil
from datetime import datetime
from importlib.machinery import SourceFileLoader
from importlib.util import module_from_spec, spec_from_loader
from pathlib import Path

from IPython.display import HTML, display

# Giữ môi trường import sạch hơn và tránh một số request phụ từ LiteLLM.
os.environ.setdefault("KAGGLE_ENVELOPES", "0")
os.environ.setdefault("LITELLM_LOCAL_MODEL_COST_MAP", "True")
os.environ.setdefault("LITELLM_LOG", "ERROR")
logging.getLogger("kaggle_environments").setLevel(logging.ERROR)
logging.getLogger("LiteLLM").setLevel(logging.ERROR)

from kaggle_environments import make
import kaggle_environments as ke

print("kaggle_environments:", getattr(ke, "__version__", "unknown"))


def load_agent_from_file(path):
    file_path = Path(path)
    module_name = f"orbit_wars_{file_path.stem}_{abs(hash(str(file_path))) & 0xffffffff}"

    loader = SourceFileLoader(module_name, str(file_path))
    spec = spec_from_loader(module_name, loader)
    module = module_from_spec(spec)

    sys.modules[module_name] = module  # dòng quan trọng

    loader.exec_module(module)

    if hasattr(module, "agent"):
        return module.agent
    if hasattr(module, "my_agent"):
        return module.my_agent

    raise AttributeError(f"{path} không có hàm agent hoặc my_agent")


def play_match(agent_files, debug=False):
    """Chạy một trận và trả về env, reward, status cuối trận."""
    agent_funcs = [load_agent_from_file(path) for path in agent_files]
    env = make("orbit_wars", debug=debug)
    env.run(agent_funcs)

    final_states = env.steps[-1]
    rewards = [state.reward for state in final_states]
    statuses = [state.status for state in final_states]
    return env, rewards, statuses


def save_replay_html(env, name=None, width=900, height=650):
    """Lưu replay ra file HTML để mở bằng trình duyệt."""
    REPLAY_DIR.mkdir(exist_ok=True)

    if name is None:
        name = datetime.now().strftime("orbit_wars_%Y%m%d_%H%M%S")

    replay_path = REPLAY_DIR / f"{name}.html"
    html = env.render(mode="html", width=width, height=height)
    replay_path.write_text(html, encoding="utf-8")

    print("Saved replay:", replay_path.resolve())
    display(HTML(f'<a href="{replay_path.as_posix()}" target="_blank">Mở replay HTML</a>'))
    return replay_path.resolve()


def promote_agent_to_main(source_file, target_file="main.py"):
    """Copy agent đã chọn thành main.py."""
    source_path = Path(source_file).resolve()
    target_path = Path(target_file).resolve()
    shutil.copyfile(source_path, target_path)
    print(f"Copied {source_path} -> {target_path}")


_ = make("orbit_wars", debug=False)
print("orbit_wars đã sẵn sàng.")


kaggle_environments: 1.29.3
orbit_wars đã sẵn sàng.


In [ ]:
# Danh sách agent muốn cho đấu.
# Đổi tên file ở đây nếu bạn thêm agent mới.

AGENT_FILES = [
    ROOT / "main_plus_main5_ideas.py",
    ROOT / "submission.py",
]

for agent_file in AGENT_FILES:
    if not agent_file.exists():
        raise FileNotFoundError(agent_file)

print("Agents:")
for agent_file in AGENT_FILES:
    print("-", agent_file.name)


Agents:
- submission.py
- main_plus_main5_ideas.py


In [13]:
# # Chạy một trận và lưu replay HTML.
# # Mở file trong thư mục replays/ bằng Chrome hoặc Edge nếu link trong notebook không mở được.

# env, rewards, statuses = play_match(AGENT_FILES, debug=False)

# print("Rewards:", rewards)
# print("Statuses:", statuses)
# print("Steps:", len(env.steps))

# replay_path = save_replay_html(env, name="last_match")
# replay_path


In [14]:
import sys
print(sys.executable)

d:\Orbit Wars\.venv\Scripts\python.exe


In [15]:
import sys
!{sys.executable} -m pip install torch

'd:\Orbit' is not recognized as an internal or external command,
operable program or batch file.


In [19]:
# Chạy nhiều trận để so điểm trung bình.
# Cell này không render trong vòng lặp để notebook không bị nặng.

from collections import defaultdict
import torch
N_MATCHES = 10
scoreboard = defaultdict(list)
last_env = None

for match_index in range(N_MATCHES):
    last_env, rewards, statuses = play_match(AGENT_FILES, debug=False)

    for agent_file, reward in zip(AGENT_FILES, rewards):
        scoreboard[Path(agent_file).stem].append(reward)

    print(f"Match {match_index + 1}/{N_MATCHES}:", rewards, statuses)

print()
for agent_name, reward_list in sorted(scoreboard.items()):
    avg_reward = sum(reward_list) / len(reward_list)
    print(agent_name, "avg_reward=", round(avg_reward, 4), "matches=", len(reward_list))

# Nếu muốn xem replay của trận cuối, bỏ comment dòng dưới.
save_replay_html(last_env, name="last_match")

Match 1/10: [1, -1] ['DONE', 'DONE']
Match 2/10: [1, -1] ['DONE', 'DONE']
Match 3/10: [1, -1] ['DONE', 'DONE']
Match 4/10: [1, -1] ['DONE', 'DONE']
Match 5/10: [-1, 1] ['DONE', 'DONE']
Match 6/10: [1, -1] ['DONE', 'DONE']
Match 7/10: [1, -1] ['DONE', 'DONE']
Match 8/10: [1, -1] ['DONE', 'DONE']
Match 9/10: [1, -1] ['DONE', 'DONE']
Match 10/10: [1, -1] ['DONE', 'DONE']

main_plus_main5_ideas avg_reward= -0.8 matches= 10
submission avg_reward= 0.8 matches= 10
Saved replay: D:\Orbit Wars\replays\last_match.html


WindowsPath('D:/Orbit Wars/replays/last_match.html')

In [17]:
# Chốt agent thắng thành main.py.
# Đổi agent_04.py thành file bạn muốn submit.

# promote_agent_to_main(ROOT / "agent_04.py")
